In [14]:
import os
import glob
import shutil
from dotenv import load_dotenv
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Protocol, runtime_checkable, Sequence, Optional, Union
import random

import numpy as np

from sklearn.manifold import TSNE

from langchain.document_loaders import DirectoryLoader
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_core.callbacks import StdOutCallbackHandler

import plotly.graph_objects as go

import gradio as gr

In [2]:
# initialize
load_dotenv(override=True)

True

In [3]:
# config
MODEL = "gpt-4o-mini"
query = "Can you describe the Insurellm in a few sentences?"

In [7]:
# ========= OOP: Strategies and Abstractions ========= #

@runtime_checkable
class DocumentLoaderStrategy(Protocol):
    def load(self, path: str) -> list[Document]:
        ...


class MarkdownDirectoryLoader(DocumentLoaderStrategy):
    """Loads .md files from immediate subfolders and tags doc_type."""
    def load(self, path: str) -> list[Document]:
        docs: list[Document] = []
        folders = glob.glob(f"{path}/*")
        if not folders:
            raise ValueError(f"No folders found in {path}")
        for folder in folders:
            if not os.path.isdir(folder):
                raise ValueError(f"{folder} is not a directory")
            doc_type = os.path.basename(folder)
            loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader)
            for doc in loader.load():
                doc.metadata = {**doc.metadata, "doc_type": doc_type}
                docs.append(doc)
        return docs


@runtime_checkable
class ChunkerStrategy(Protocol):
    def split(self, documents: list[Document]) -> list[Document]:
        ...


@dataclass
class CharacterChunker(ChunkerStrategy):
    chunk_size: int = 1000
    chunk_overlap: int = 200

    def split(self, documents: list[Document]) -> list[Document]:
        if not documents:
            return []
        splitter = CharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
        )
        return splitter.split_documents(documents)


@runtime_checkable
class EmbeddingProvider(Protocol):
    def create(self):  # returns an embedding object compatible with vector stores
        ...


class OpenAIEmbeddingProvider(EmbeddingProvider):
    def create(self):
        return OpenAIEmbeddings()


# ========= Vector Store Strategy ========= #

@runtime_checkable
class VectorStore(Protocol):
    """Common interface for vector stores (Chroma, FAISS, etc.)."""
    def get(self, include: list[str] = None, **kwargs) -> dict:
        ...
    
    def similarity_search(self, query: str, k: int = 4, **kwargs) -> list[Document]:
        ...


@dataclass
class VectorStoreStrategy(Protocol):
    """Strategy for creating different vector store backends."""
    def create_from_documents(
        self, documents: list[Document], embedding
    ) -> Union[Chroma, FAISS]:
        ...


@dataclass
class ChromaStoreStrategy:
    """Chroma-specific vector store creation with persistence and fallbacks."""
    persist_directory: str = "chroma_db"
    collection_name: str = "company_docs"

    def _prepare_directory(self, path: str) -> None:
        """Ensure a clean, writable directory exists at path."""
        if os.path.exists(path):
            shutil.rmtree(path)
        os.makedirs(path, exist_ok=True)
        # quick writability probe
        test_file = os.path.join(path, ".writetest")
        with open(test_file, "w") as f:
            f.write("ok")
        os.remove(test_file)

    def create_from_documents(
        self, documents: list[Document], embedding
    ) -> Chroma:
        if not documents:
            raise ValueError("No documents provided to ChromaStoreStrategy")

        # Prepare the target directory
        try:
            self._prepare_directory(self.persist_directory)
        except Exception as e:
            raise RuntimeError(
                f"Failed to prepare persist directory '{self.persist_directory}': {e}"
            ) from e

        # Attempt persistent creation; fall back on common environment/version issues
        try:
            store = Chroma.from_documents(
                documents=documents,
                embedding=embedding,
                persist_directory=self.persist_directory,
                collection_name=self.collection_name,
            )
        except Exception as e:
            msg = str(e).lower()
            if "readonly" in msg or "read-only" in msg:
                import tempfile, uuid
                alt_dir = os.path.join(tempfile.gettempdir(), f"chroma_db_{uuid.uuid4().hex}")
                self._prepare_directory(alt_dir)
                store = Chroma.from_documents(
                    documents=documents,
                    embedding=embedding,
                    persist_directory=alt_dir,
                    collection_name=self.collection_name,
                )
            elif "no such table: tenants" in msg:
                # Version/schema mismatch in local persistent backend; use ephemeral client
                import chromadb
                store = Chroma.from_documents(
                    documents=documents,
                    embedding=embedding,
                    client=chromadb.Client(),  # in-memory; no persistence
                    collection_name=self.collection_name,
                )
            else:
                raise

        if hasattr(store, "persist"):
            try:
                store.persist()
            except Exception:
                # Some clients may not support persist(); ignore to keep flow running
                pass
        return store


@dataclass
class FAISSStoreStrategy:
    """FAISS-specific vector store creation with optional persistence."""
    index_path: Optional[str] = "faiss_index"
    
    def create_from_documents(
        self, documents: list[Document], embedding
    ) -> FAISS:
        if not documents:
            raise ValueError("No documents provided to FAISSStoreStrategy")
        
        # Create FAISS index from documents
        store = FAISS.from_documents(
            documents=documents,
            embedding=embedding,
        )
        
        # Optionally save to disk
        if self.index_path:
            try:
                # Ensure parent directory exists
                index_dir = os.path.dirname(self.index_path) if os.path.dirname(self.index_path) else "."
                os.makedirs(index_dir, exist_ok=True)
                store.save_local(self.index_path)
            except Exception as e:
                # If save fails, continue with in-memory index
                print(f"Warning: Could not save FAISS index to {self.index_path}: {e}")
        
        return store


# ========= Factory & Facade ========= #

@dataclass
class VectorStoreFactory:
    """Factory for creating vector stores with different backends."""
    backend: str = "chroma"  # "chroma" or "faiss"
    embedding_provider: EmbeddingProvider = field(default_factory=OpenAIEmbeddingProvider)
    persist_directory: str = "chroma_db"
    collection_name: str = "company_docs"
    index_path: str = "faiss_index"

    def create_from_documents(self, documents: list[Document]) -> Union[Chroma, FAISS]:
        if not documents:
            raise ValueError("No documents provided to VectorStoreFactory")
        
        embeddings = self.embedding_provider.create()
        
        if self.backend.lower() == "chroma":
            strategy = ChromaStoreStrategy(
                persist_directory=self.persist_directory,
                collection_name=self.collection_name,
            )
            return strategy.create_from_documents(documents, embeddings)
        elif self.backend.lower() == "faiss":
            strategy = FAISSStoreStrategy(index_path=self.index_path)
            return strategy.create_from_documents(documents, embeddings)
        else:
            raise ValueError(f"Unsupported backend: {self.backend}. Choose 'chroma' or 'faiss'.")


class EmbeddingVisualizer:
    def __init__(self, dims: int = 2):
        if dims not in (2, 3):
            raise ValueError("dims must be 2 or 3")
        self.dims = dims

    def plot(self, collection: Union[Chroma, FAISS]) -> go.Figure:
        # Handle both Chroma and FAISS
        if isinstance(collection, FAISS):
            # FAISS: extract embeddings from index
            index_to_docstore_id = collection.index_to_docstore_id
            embeddings_list = []
            metadatas_list = []
            for idx in sorted(index_to_docstore_id.keys()):
                doc_id = index_to_docstore_id[idx]
                doc = collection.docstore.search(doc_id)
                if doc:
                    metadatas_list.append(doc.metadata)
                    # Reconstruct embedding from index
                    emb = collection.index.reconstruct(int(idx))
                    embeddings_list.append(emb)
            embs = embeddings_list
            metas = metadatas_list
        else:
            # Chroma or other VectorStore with .get()
            data = collection.get(include=["embeddings", "metadatas"])
            embs = data.get("embeddings")
            metas = data.get("metadatas") or []
        
        if embs is None or len(embs) == 0:
            raise ValueError("No embeddings found in collection")
        
        embs_np = np.array(embs, dtype=np.float32)
        if embs_np.size == 0:
            raise ValueError("No embeddings found in collection")
        if embs_np.ndim == 1:
            embs_np = embs_np.reshape(-1, 1)

        tsne = TSNE(n_components=self.dims, random_state=42)
        proj = tsne.fit_transform(embs_np)

        doc_types = [m.get("doc_type", "unknown") for m in metas]
        unique_types = sorted(set(doc_types))

        random.seed(42)
        def random_hex_color():
            return "#{:06x}".format(random.randint(0, 0xFFFFFF))
        palette = [random_hex_color() for _ in unique_types]
        mapping = {t: palette[i] for i, t in enumerate(unique_types)}
        colors = [mapping.get(t, "#888888") for t in doc_types]

        if self.dims == 2:
            fig = go.Figure(
                data=go.Scatter(
                    x=proj[:, 0], y=proj[:, 1], mode="markers",
                    marker=dict(color=colors, size=8, opacity=0.7),
                    text=doc_types, hoverinfo="text",
                )
            )
            fig.update_layout(
                title="t-SNE Visualization of Document Embeddings (2D)",
                xaxis_title="t-SNE Dimension 1",
                yaxis_title="t-SNE Dimension 2",
                width=800, height=600,
                margin=dict(r=20, b=10, l=10, t=40),
            )
            return fig
        fig = go.Figure(
            data=go.Scatter3d(
                x=proj[:, 0], y=proj[:, 1], z=proj[:, 2], mode="markers",
                marker=dict(color=colors, size=6, opacity=0.7),
                text=doc_types, hoverinfo="text",
            )
        )
        fig.update_layout(
            title="t-SNE Visualization of Document Embeddings (3D)",
            scene=dict(
                xaxis_title="t-SNE Dimension 1",
                yaxis_title="t-SNE Dimension 2",
                zaxis_title="t-SNE Dimension 3",
            ),
            width=900, height=700,
            margin=dict(r=20, b=10, l=10, t=40),
        )
        return fig


@dataclass
class IngestionPipeline:
    loader: DocumentLoaderStrategy
    chunker: ChunkerStrategy
    factory: VectorStoreFactory

    def run(self, source_path: str) -> Union[Chroma, FAISS]:
        docs = self.loader.load(source_path)
        chunks = self.chunker.split(docs)
        return self.factory.create_from_documents(chunks)


# Example usage with Chroma (default):
print("=== Creating Chroma collection ===")
pipeline_chroma = IngestionPipeline(
    loader=MarkdownDirectoryLoader(),
    chunker=CharacterChunker(chunk_size=1000, chunk_overlap=200),
    factory=VectorStoreFactory(backend="chroma", persist_directory="chroma_db", collection_name="company_docs"),
)
vector_store = pipeline_chroma.run("knowledge-base")
viz_chroma = EmbeddingVisualizer(dims=2)
fig_chroma = viz_chroma.plot(vector_store)
fig_chroma

Created a chunk of size 1088, which is longer than the specified 1000


=== Creating Chroma collection ===


In [10]:
# Conversational Retrieval Chain setup
llm = ChatOpenAI(temperature=0.7, model_name=MODEL)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
retriever = vector_store.as_retriever()
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm, retriever=retriever, memory=memory
)

In [ ]:
def chat(message, history):
    result = conversation_chain.invoke({"question": message})
    return result["answer"]

In [ ]:
# view = gr.ChatInterface(
#     fn=chat,
#     type="messages"
# ).launch(debug=True)

In [12]:
# Example usage with FAISS:
print("=== Creating FAISS collection ===")
pipeline_faiss = IngestionPipeline(
    loader=MarkdownDirectoryLoader(),
    chunker=CharacterChunker(chunk_size=1000, chunk_overlap=200),
    factory=VectorStoreFactory(backend="faiss", index_path="faiss_index"),
)
vector_store = pipeline_faiss.run("knowledge-base")
viz_faiss = EmbeddingVisualizer(dims=2)
fig_faiss = viz_faiss.plot(vector_store)
fig_faiss

Created a chunk of size 1088, which is longer than the specified 1000


=== Creating FAISS collection ===


In [ ]:
llm = ChatOpenAI(temperature=0.7, model_name=MODEL)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
retriever = vector_store.as_retriever(search_kwargs={"k":25})
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm, retriever=retriever, memory=memory, callbacks=[StdOutCallbackHandler()]
)

query = "Who received the prestigious IIOTY award in 2023?"
result = conversation_chain.invoke({"question": query})
print(f"Q: {query}\nA: {result['answer']}")



> Entering new ConversationalRetrievalChain chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
- **2022**: **Satisfactory**  
  Avery focused on rebuilding team dynamics and addressing employee concerns, leading to overall improvement despite a saturated market.  

- **2023**: **Exceeds Expectations**  
  Market leadership was regained with innovative approaches to personalized insurance solutions. Avery is now recognized in industry publications as a leading voice in Insurance Tech innovation.

## Annual Performance History
- **2020:**  
  - Completed onboarding successfully.  
  - Met expectations in delivering project milestones.  
  - Received positive feedback from the team leads.

- **2021:**  
  - Achieved a 95% success rate in proje